# Agentic Data Visualization — Kaggle + Hugging Face

A reproducible **Planner → Data Tool → Coder → Reviewer → deterministic code validator → Plotly Executor** workflow. Rejected code and execution errors are passed back to Coder in a **bounded correction loop** (maximum 3 attempts by default). This extends the original Planner/Tools/Coder/Executor design rather than replacing it.

- **DEMO_MODE=True**: deterministic stand-ins for Planner/Coder/Reviewer to verify the **full loop and chart execution** without an API key. The demo is fixed to monthly revenue and does **not** represent live AI-agent reasoning.
- **DEMO_MODE=False**: live Planner, Coder and independent Reviewer requests through Hugging Face `InferenceClient`. Requires a valid HF token and currently available hosted models.
- **AUTO_EXECUTE=True**: approved code is automatically executed after static restrictions; output appears immediately.

**Execution warning:** static AST allowlisting and an LLM Reviewer are **not a secure sandbox**. Run this notebook on trusted data and in a disposable Colab runtime, not with sensitive credentials or broad file access.

In [ ]:
%pip -q install kagglehub pandas plotly huggingface_hub

In [ ]:
from pathlib import Path
import os
import re
import json
import ast
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import kagglehub
from huggingface_hub import InferenceClient

# DEMO runs the ENTIRE pipeline with deterministic stand-ins for the LLM roles.
# LIVE uses hosted LLMs for Planner, Coder and Reviewer.
DEMO_MODE = True
AUTO_EXECUTE = True     # Only after Reviewer approval AND deterministic code validation
MAX_ATTEMPTS = 3       # Includes initial Coder attempt; no infinite loops

# Original project choices; hosted provider access/model IDs must be verified at run time.
PLANNER_MODEL = "Qwen/Qwen2.5-72B-Instruct"
CODER_MODEL = "deepseek-ai/DeepSeek-V4-Pro"
REVIEWER_MODEL = PLANNER_MODEL

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except (ImportError, KeyError):
        return None

if DEMO_MODE:
    client = None
else:
    hf_token = get_hf_token()
    if not hf_token:
        raise RuntimeError("LIVE mode requires an HF_TOKEN from Colab Secrets or env vars")
    client = InferenceClient(token=hf_token)

print("Mode:", "DEMO (no LLM API)" if DEMO_MODE else "LIVE Hugging Face agents")
print("Auto-execution:", AUTO_EXECUTE, "| Maximum attempts:", MAX_ATTEMPTS)

In [ ]:
# Kagglehub chooses the correct local cache path; do not hard-code a version directory.
dataset_ref = "ssssws/chocolate-sales-dataset-2023-2024"
dataset_path = Path(kagglehub.dataset_download(dataset_ref))
required_files = ["sales.csv", "products.csv", "stores.csv", "customers.csv"]
missing = [f for f in required_files if not (dataset_path / f).is_file()]
if missing:
    raise FileNotFoundError(f"Missing Kaggle CSVs: {missing}; found: {[x.name for x in dataset_path.iterdir()]}")

sales_df = pd.read_csv(dataset_path / "sales.csv")
products_df = pd.read_csv(dataset_path / "products.csv")
stores_df = pd.read_csv(dataset_path / "stores.csv")
customer_df = pd.read_csv(dataset_path / "customers.csv")
master_df = (sales_df.merge(products_df, on="product_id")
    .merge(stores_df, on="store_id")
    .merge(customer_df, on="customer_id"))
master_df["order_date"] = pd.to_datetime(master_df["order_date"], errors="coerce")

print("Merged data shape:", master_df.shape)
display(master_df.head(3))

## Architecture and source-data inspection
`master_df` merges `sales.csv`, `products.csv`, `stores.csv`, `customers.csv`. The Planner selects either `get_metadata` or `get_sample`. Coder sees the selected tool output and the exact column names. Reviewer checks semantic correctness, then deterministic validation blocks unsupported Python syntax and obvious I/O. Errors feed back into Coder for corrections.

In [ ]:
metadata = {"columns": master_df.columns.tolist(),
            "dtypes": master_df.dtypes.astype(str).to_dict(),
            "sample": master_df.head(3).to_dict(orient="records")}
print("Available columns:", metadata["columns"])

## Agent pipeline with Reviewer + bounded correction loop
The original architecture is retained and a Reviewer is added as a new explicit step. In live mode Reviewer is a separate LLM call (its model ID is configurable). An independent deterministic AST validator runs before and again at the execution boundary.

In [ ]:
# -- AGENT PIPELINE --
# Original responsibilities preserved: Planner -> DataTools -> Coder -> Plotly.
# Added: Reviewer + deterministic validator + bounded correction loop.
# WARNING: AST allowlisting is defense in depth, NOT a secure sandbox for untrusted Python.

class CodeRejected(ValueError):
    """Generated code failed deterministic validation."""

class ChartCodeValidator:
    """Strict, illustrative subset of pandas/Plotly Python; no imports, loops or I/O."""

    MAX_CHARS = 8000
    MAX_NODES = 300
    ALLOWED_NODES = (
        ast.Module, ast.Assign, ast.Expr, ast.Name, ast.Load, ast.Store,
        ast.Attribute, ast.Call, ast.keyword, ast.Constant, ast.Subscript,
        ast.Slice, ast.List, ast.Tuple, ast.Dict, ast.BinOp, ast.UnaryOp,
        ast.Compare, ast.BoolOp, ast.IfExp,
        ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Mod, ast.Pow,
        ast.UAdd, ast.USub, ast.Invert, ast.BitAnd, ast.BitOr,
        ast.Eq, ast.NotEq, ast.Lt, ast.LtE, ast.Gt, ast.GtE,
        ast.And, ast.Or, ast.Is, ast.IsNot, ast.In, ast.NotIn,
    )
    # Data transformation and graph methods supported by this notebook's code contract.
    ALLOWED_METHODS = {
        "copy", "dropna", "fillna", "astype", "groupby", "agg", "sum", "mean",
        "median", "count", "size", "reset_index", "rename", "sort_values",
        "value_counts", "head", "tail", "nlargest", "nsmallest", "round",
        "pivot_table", "unstack", "to_period", "to_timestamp", "isin", "isna",
        "drop_duplicates", "between", "clip", "dt", "str", "columns", "loc", "iloc",
        "line", "bar", "scatter", "histogram", "pie", "area", "box", "violin",
        "density_heatmap", "to_datetime", "update_layout", "update_traces",
        "add_hline", "add_vline", "add_scatter", "add_bar", "add_trace",
    }
    ALLOWED_BUILTINS = {"len": len, "str": str, "float": float,
                        "int": int, "round": round, "abs": abs, "min": min,
                        "max": max, "sum": sum}
    READONLY = {"master_df", "pd", "px", "__builtins__"}
    MAX_ROWS = 200000

    @classmethod
    def validate(cls, generated_code, df):
        if not isinstance(generated_code, str) or not generated_code.strip():
            raise CodeRejected("Empty generated code")
        if len(generated_code) > cls.MAX_CHARS:
            raise CodeRejected("Code exceeds the length limit")
        if len(df) > cls.MAX_ROWS:
            raise CodeRejected(f"Dataset has {len(df)} rows; demo executor limit is {cls.MAX_ROWS}")
        try:
            tree = ast.parse(generated_code)
        except SyntaxError as err:
            raise CodeRejected(f"Python syntax error: {err.msg} (line {err.lineno})") from err
        nodes = list(ast.walk(tree))
        if len(nodes) > cls.MAX_NODES:
            raise CodeRejected("Code exceeds the AST size limit")

        has_fig_assignment = False
        for node in nodes:
            if not isinstance(node, cls.ALLOWED_NODES):
                raise CodeRejected(f"Disallowed Python syntax: {type(node).__name__}")
            if isinstance(node, ast.Name) and node.id.startswith("_"):
                raise CodeRejected(f"Private/dunder name not permitted: {node.id}")
            if isinstance(node, ast.Attribute):
                if node.attr.startswith("_") or node.attr not in cls.ALLOWED_METHODS:
                    raise CodeRejected(f"Unsupported or unsafe attribute: {node.attr}")
            if isinstance(node, ast.Call):
                f = node.func
                if isinstance(f, ast.Name):
                    if f.id not in cls.ALLOWED_BUILTINS:
                        raise CodeRejected(f"Function call not permitted: {f.id}")
                elif not isinstance(f, ast.Attribute):
                    raise CodeRejected("Only approved method/function calls are permitted")
            if isinstance(node, ast.Assign):
                for target in node.targets:
                    if isinstance(target, ast.Name):
                        if target.id in cls.READONLY:
                            raise CodeRejected(f"Cannot overwrite reserved variable: {target.id}")
                        has_fig_assignment |= target.id == "fig"
                    elif isinstance(target, ast.Subscript):
                        root = target.value
                        if isinstance(root, ast.Name) and root.id in cls.READONLY:
                            raise CodeRejected("Cannot mutate master_df or other reserved objects")
                    else:
                        raise CodeRejected("Only simple assignments and local DataFrame columns supported")
            # Check direct references to dataset columns, not derived intermediary fields.
            if isinstance(node, ast.Subscript) and isinstance(node.value, ast.Name):
                if node.value.id == "master_df" and isinstance(node.slice, ast.Constant) and isinstance(node.slice.value, str):
                    if node.slice.value not in df.columns:
                        raise CodeRejected(f"Unknown column in master_df: {node.slice.value}")
        if not has_fig_assignment:
            raise CodeRejected("Code must assign a Plotly figure to variable 'fig'")
        return tree


class DataTools:
    @staticmethod
    def get_metadata(df, columns=None):
        if columns:
            cols = [str(c) for c in columns if str(c) in df.columns]
            return {c: str(df[c].dtype) for c in cols}
        return df.dtypes.astype(str).to_dict()

    @staticmethod
    def get_sample(df, columns=None):
        if isinstance(columns, dict):
            columns = list(columns.keys())
        cols = [str(c) for c in (columns or df.columns.tolist()) if str(c) in df.columns]
        return df[cols].head(3).to_dict(orient="records")

    @staticmethod
    def execute_viz(df, generated_code, show=True):
        """Validate again at the execution boundary, execute, then show a Plotly figure."""
        tree = ChartCodeValidator.validate(generated_code, df)
        scope = {
            "__builtins__": ChartCodeValidator.ALLOWED_BUILTINS,
            "master_df": df.copy(deep=True), "pd": pd, "px": px,
        }
        # WARNING: In-process Python execution is not a security isolation boundary.
        exec(compile(tree, filename="<validated-plotly>", mode="exec"), scope, scope)
        fig = scope.get("fig")
        if not isinstance(fig, go.Figure) or not fig.data:
            raise ValueError("Execution produced no nonempty Plotly figure in 'fig'")
        if show:
            fig.show()  # Trusted display, not LLM-generated fig.show()
        return fig


class VisualAgent:
    def __init__(self, df, client=None, demo_mode=True, max_attempts=3, auto_execute=True):
        if not 1 <= max_attempts <= 5:
            raise ValueError("max_attempts must be between 1 and 5")
        self.df = df
        self.client = client
        self.demo_mode = demo_mode
        self.max_attempts = max_attempts
        self.auto_execute = auto_execute
        self.history = []
        self.available_tools = {
            "get_metadata": DataTools.get_metadata,
            "get_sample": DataTools.get_sample,
        }
        if not demo_mode and client is None:
            raise ValueError("LIVE mode needs a configured Hugging Face InferenceClient")

    def _chat(self, model, prompt, max_tokens):
        response = self.client.chat_completion(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
        )
        return response.choices[0].message.content or ""

    @staticmethod
    def _parse_json(raw):
        cleaned = re.sub(r"<think>.*?</think>", "", raw, flags=re.S)
        fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", cleaned, flags=re.S | re.I)
        candidate = fenced.group(1) if fenced else None
        if candidate is None:
            left, right = cleaned.find("{"), cleaned.rfind("}")
            if left < 0 or right <= left:
                raise ValueError("Response did not contain a JSON object")
            candidate = cleaned[left:right + 1]
        value = json.loads(candidate)
        if not isinstance(value, dict):
            raise ValueError("Expected a JSON object")
        return value

    @staticmethod
    def _extract_code(raw):
        cleaned = re.sub(r"<think>.*?</think>", "", raw, flags=re.S)
        match = re.search(r"```(?:python)?\s*(.*?)\s*```", cleaned, flags=re.S | re.I)
        code = match.group(1).strip() if match else cleaned.strip()
        if not code:
            raise ValueError("Coder returned empty code")
        return code

    def planner_step(self, question):
        if self.demo_mode:
            return {
                "tool_name": "get_sample",
                "tool_args": [c for c in ["order_date", "revenue", "profit"] if c in self.df.columns],
                "viz_goal": "Monthly revenue trend using existing order_date/revenue columns",
            }
        prompt = f"""You are the Planner Agent for pandas/Plotly visualization.
DataFrame column types: {DataTools.get_metadata(self.df)}
Allowed tools: {list(self.available_tools)}
User question: {question}
Choose exactly one data-inspection tool. Return ONLY JSON object with keys:
{{"tool_name":"get_sample or get_metadata", "tool_args":["column_name"], "viz_goal":"specific visualization objective"}}.
Never invent column names."""
        plan = self._parse_json(self._chat(PLANNER_MODEL, prompt, max_tokens=600))
        if plan.get("tool_name") not in self.available_tools:
            raise ValueError(f"Planner chose unknown tool: {plan.get('tool_name')}")
        if not isinstance(plan.get("viz_goal"), str) or not plan["viz_goal"].strip():
            raise ValueError("Planner provided no visualization goal")
        if not isinstance(plan.get("tool_args"), list):
            plan["tool_args"] = []
        # Eliminate invented columns before calling tools.
        plan["tool_args"] = [str(c) for c in plan["tool_args"] if str(c) in self.df.columns]
        return plan

    def coder_step(self, plan, observation, feedback="", previous_code=""):
        if self.demo_mode:
            if not {"order_date", "revenue"}.issubset(self.df.columns):
                raise ValueError("Demo requires order_date and revenue columns")
            return """chart_df = master_df.dropna(subset=["order_date"]).copy()
chart_df["month"] = chart_df["order_date"].dt.to_period("M").astype(str)
monthly = chart_df.groupby("month", as_index=False)["revenue"].sum()
fig = px.line(monthly, x="month", y="revenue", markers=True, title="Monthly Revenue Trend (Demo)")"""

        prompt = f"""You are the Coder Agent. Produce a self-contained Plotly Express Python snippet.
Existing DataFrame is master_df. Exact columns/types: {DataTools.get_metadata(self.df)}
User visualization goal: {plan['viz_goal']}
Tool output (data preview): {str(observation)[:3000]}
Previous attempt code (if any): {previous_code[:5000]}
Reviewer/validator/executor error feedback (correct it): {feedback[:1800]}
Rules:
- Use ONLY existing master_df columns; you may create local derived DataFrames/columns.
- Use pandas & px only. No imports, filesystem, network, eval, exec, loops, custom functions.
- Allowed Plotly Express charts: line, bar, scatter, histogram, pie, area, box, violin, density_heatmap.
- Use .reset_index() after groupby unless as_index=False; convert monthly periods to strings.
- Assign a nonempty Plotly Figure to variable fig. Do NOT call fig.show().
- Output ONLY ```python ... ``` with executable code; do not invent columns."""
        return self._extract_code(self._chat(CODER_MODEL, prompt, max_tokens=1200))

    def reviewer_step(self, question, plan, observation, generated_code):
        """Independent semantic review plus deterministic static validation."""
        try:
            ChartCodeValidator.validate(generated_code, self.df)
        except (CodeRejected, ValueError) as err:
            return {"approved": False, "feedback": f"Static validator rejected code: {err}", "source": "validator"}

        if self.demo_mode:
            return {"approved": True, "feedback": "Static validation passed (DEMO: no LLM reviewer)", "source": "demo-validator"}

        prompt = f"""You are an independent Reviewer Agent. Evaluate the Coder's Plotly Python snippet.
Original user request: {question}
Planner goal: {plan['viz_goal']}
Available dataset column types: {DataTools.get_metadata(self.df)}
Tool observation: {str(observation)[:2000]}
Candidate code:
```python
{generated_code}
```
Check: matching user intent, real columns, correct aggregations/units, chart axes, handling date and null values,
and logical errors. If the dataset lacks a requested field, require a clear fallback or fail.
Do not approve based only on valid syntax. Return ONLY JSON:
{{"approved": true, "feedback": "short reason"}} or
{{"approved": false, "feedback": "specific code corrections"}}.
"""
        try:
            review = self._parse_json(self._chat(REVIEWER_MODEL, prompt, max_tokens=600))
            if type(review.get("approved")) is not bool:
                raise ValueError("Reviewer 'approved' must be boolean")
            feedback = review.get("feedback", "")
            if not isinstance(feedback, str):
                feedback = str(feedback)
            if not review["approved"] and not feedback.strip():
                feedback = "Reviewer rejected the code without detailed reasons"
            return {"approved": review["approved"], "feedback": feedback[:1500], "source": "llm-reviewer"}
        except Exception as err:
            # No silent approval if reviewer API fails or returns malformed JSON.
            return {"approved": False, "feedback": f"Reviewer unavailable/invalid response: {err}", "source": "llm-reviewer"}

    def run_once(self, question, show=True):
        if not isinstance(question, str) or not question.strip():
            raise ValueError("Please provide a visualization question")
        print("🧠 Planner Agent...")
        plan = self.planner_step(question)
        # Whitelisted dynamic dispatch: only the two pre-existing DataTools are callable.
        tool = self.available_tools[plan["tool_name"]]
        observation = tool(self.df, plan.get("tool_args", []))
        print(f"🛠️ Tool: {plan['tool_name']} | Goal: {plan['viz_goal']}")

        feedback, previous_code = "", ""
        attempts = []
        for attempt in range(1, self.max_attempts + 1):
            print(f"💻 Coder Agent: attempt {attempt}/{self.max_attempts}")
            try:
                candidate = self.coder_step(plan, observation, feedback, previous_code)
                print("🔎 Reviewer Agent + static validation...")
                review = self.reviewer_step(question, plan, observation, candidate)
                if not review["approved"]:
                    feedback = review["feedback"]
                    attempts.append({"attempt": attempt, "status": "rejected", "feedback": feedback})
                    previous_code = candidate
                    print("🔄 Needs correction:", feedback)
                    continue
                if not self.auto_execute:
                    attempts.append({"attempt": attempt, "status": "approved_not_executed"})
                    result = {"plan": plan, "code": candidate, "figure": None,
                              "attempts": attempts, "status": "approved_not_executed"}
                    self.history.append({"question": question, "result": result["status"], "attempts": attempt})
                    return result
                print("📊 Executing validated code...")
                fig = DataTools.execute_viz(self.df, candidate, show=show)
                attempts.append({"attempt": attempt, "status": "success"})
                result = {"plan": plan, "code": candidate, "figure": fig,
                          "attempts": attempts, "status": "success"}
                self.history.append({"question": question, "result": "success", "attempts": attempt})
                print(f"✅ Visualization completed on attempt {attempt}")
                return result
            except Exception as err:
                # Provide compact error details to Coder, but retry only a bounded number of times.
                feedback = f"{type(err).__name__}: {str(err)[:1200]}"
                attempts.append({"attempt": attempt, "status": "error", "feedback": feedback})
                previous_code = locals().get("candidate", "")
                print("🔄 Execution/generation failed:", feedback)
        self.history.append({"question": question, "result": "failed", "attempts": len(attempts)})
        raise RuntimeError(f"Agent stopped after {self.max_attempts} attempts. Last issue: {feedback}")

    def run(self):
        """Original interactive loop, optional in Colab (call agent.run())."""
        print("🤖 Agentic visualization ready. Type 'exit' to quit.")
        while True:
            question = input("Question: ").strip()
            if question.lower() in {"exit", "quit"}:
                break
            try:
                self.run_once(question)
            except Exception as err:
                print("⚠️", err)

## Run the end-to-end pipeline
In DEMO mode, the whole workflow executes automatically using a fixed monthly revenue chart. After verifying it, switch to LIVE mode to test with Hugging Face agents, a valid token, and model IDs available through your provider.

In [ ]:
question = "Show the monthly revenue trend for chocolate sales."
agent = VisualAgent(master_df, client=client, demo_mode=DEMO_MODE,
                    max_attempts=MAX_ATTEMPTS, auto_execute=AUTO_EXECUTE)
result = agent.run_once(question, show=True)
print("Status:", result["status"])
print("Attempt log:", result["attempts"])
# To ask many questions in an input loop, run: agent.run()

## Live mode, limitations, and safe re-execution
1. Save a valid `HF_TOKEN` in **Google Colab Secrets** and allow this notebook access (or set an environment variable in your own environment).
2. Set `DEMO_MODE=False` in the setup cell. Make sure `PLANNER_MODEL`, `CODER_MODEL`, and `REVIEWER_MODEL` are supported for your account/provider **today**; the IDs shown are references from the original notebook and may not be available.
3. Run cells top-to-bottom; the pipeline uses **Planner → Tool → Coder → Reviewer → Validator → Executor**, with maximum `MAX_ATTEMPTS` cycles. Successful charts appear automatically. When rejected, the specific feedback is passed to the next Coder attempt.
4. The AST validator only permits a deliberately narrow subset of pandas/Plotly; a legitimate generated chart may be rejected. The Coder should adapt it or the allowed operations should be reviewed by a developer.
5. A reviewer that approves code does **not** guarantee logical correctness. AST checks and in-process execution do **not** provide safe isolation against hostile code. Use a disposable runtime with limited credentials; for high-risk or multi-user deployments, move execution to a proper sandbox with resource and network restrictions.
6. Neither the dataset files nor original API tokens are stored in this repository. Older keys embedded in the original notebook should be revoked.